In [1]:
import os
import json
import pandas as pd
import numpy as np
import ast

# --- Configuration ---
data_dir = '/media/hcv530/T7/garment_folding_data/real_world/'

method_label = {
    'PlaNet-ClothPick': 'planet_clothpick_single_picker_single_primitive_multi_longsleeve_flattening_longsleeve_all_garment_5k_eps',
    'LaGarNet': 'lagarnet_single_picker_single_primitive_multi_longsleeve_flattening_sanity_check', 
    'Human': 'real_world_single_arm_human_flattening' 
}

garment_label = {
    'Longsleeve': 'small-blue-top',
    'Trousers': 'small-blue-trouser',
    'Dress': 'small-blue-dress',
    'Skirt': 'small-yellow-skirt',
    'OOD Dress': 'small-yellow-dress',

}

id_to_category = {v: k for k, v in garment_label.items()}
steps_to_evaluate = [10, 20] 

results = {m: {s: {g: {'NC': [], 'NI': [], 'MaxIoU': [], 'success': 0, 'total': 0} 
                   for g in list(garment_label.keys()) + ['Real Total']} 
               for s in steps_to_evaluate} 
           for m in method_label.keys()}

trial_counts = {m: {g: [] for g in garment_label.keys()} for m in method_label.keys()}

# ---> NEW: Data Structure for LaGarNet Timings <---
lagarnet_timings = {
    'perception': [],
    'inference': [],
    'conversion': [],
    'execution': []
}

# --- Step 1: Parse the Data ---
for method, folder_name in method_label.items():
    performance_dir = os.path.join(data_dir, folder_name, 'eval_checkpoint_-2') 
    csv_path = os.path.join(performance_dir, 'performance.csv')
    
    if not os.path.exists(csv_path):
        print(f"Warning: Could not find CSV for {method} at {csv_path}")
        continue
        
    df = pd.read_csv(csv_path)
    
    for index, row in df.iterrows():
        episode_id = row['episode_id']
        episode_folder = os.path.join(performance_dir, f'episode_{episode_id}')
        
        # Look up garment_id from info.json
        info_json_path = os.path.join(episode_folder, 'step_0', 'info.json')
        try:
            with open(info_json_path, 'r') as f:
                info_data = json.load(f)
                garment_id = info_data.get('garment_id')
                category = id_to_category.get(garment_id)
        except Exception as e:
            print(f"Skipping episode {episode_id} for {method}: Could not read info.json - {e}")
            continue
            
        if not category:
            continue 

        trial_counts[method][category].append(episode_id)

        # ---> NEW: Parse timing_report.csv for LaGarNet <---
        if method == 'LaGarNet':
            timing_csv_path = os.path.join(episode_folder, 'timing_report.csv')
            if os.path.exists(timing_csv_path):
                timing_df = pd.read_csv(timing_csv_path)
                # Append all step times from this episode
                lagarnet_timings['perception'].extend(timing_df['Perception (s)'].tolist())
                lagarnet_timings['inference'].extend(timing_df['Policy Inference (s)'].tolist())
                lagarnet_timings['conversion'].extend(timing_df['Process Action (s)'].tolist())
                lagarnet_timings['execution'].extend(timing_df['Primitives Exec (s)'].tolist())
            else:
                print(f"Warning: Could not find timing_report.csv for {method} episode {episode_id}")

        # Parse stringified lists
        nc_list = ast.literal_eval(row['evaluation/normalised_coverage'])
        ni_list = ast.literal_eval(row['evaluation/normalised_improvement'])
        iou_list = ast.literal_eval(row['evaluation/max_IoU_to_flattened'])
        
        for step in steps_to_evaluate:
            idx = min(step, len(nc_list) - 1)
            
            nc_val = max(nc_list[:idx+1]) * 100
            ni_val = max(ni_list[:idx+1]) * 100
            iou_val = max(iou_list[:idx+1]) * 100
            
            is_85_75 = 1 if any(nc_list[i] >= 0.85 and iou_list[i] >= 0.75 for i in range(idx + 1)) else 0
           
            for cat in [category, 'Real Total']:
                results[method][step][cat]['NC'].append(nc_val)
                results[method][step][cat]['NI'].append(ni_val)
                results[method][step][cat]['MaxIoU'].append(iou_val)
                results[method][step][cat]['success'] += is_85_75
                results[method][step][cat]['total'] += 1

# --- Step 1.5: Print the Trial Counts and Episode IDs ---
print("="*60)
print("TRIAL COUNTS & EPISODE IDs PER METHOD & GARMENT")
print("="*60)
for method, categories in trial_counts.items():
    print(f"\n{method}:")
    total_method_trials = 0
    for category, ep_ids in categories.items():
        count = len(ep_ids)
        print(f"  - {category}: {count} episodes (IDs: {sorted(ep_ids)})")
        total_method_trials += count
    print(f"  > Total for {method}: {total_method_trials} episodes")
print("\n" + "="*60 + "\n")

TRIAL COUNTS & EPISODE IDs PER METHOD & GARMENT

PlaNet-ClothPick:
  - Longsleeve: 10 episodes (IDs: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19])
  - Trousers: 10 episodes (IDs: [30, 31, 32, 33, 34, 35, 36, 37, 38, 39])
  - Dress: 10 episodes (IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
  - Skirt: 10 episodes (IDs: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29])
  - OOD Dress: 10 episodes (IDs: [40, 41, 42, 43, 44, 45, 46, 47, 48, 49])
  > Total for PlaNet-ClothPick: 50 episodes

LaGarNet:
  - Longsleeve: 10 episodes (IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
  - Trousers: 10 episodes (IDs: [30, 31, 32, 33, 34, 35, 36, 37, 38, 39])
  - Dress: 10 episodes (IDs: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19])
  - Skirt: 10 episodes (IDs: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29])
  - OOD Dress: 10 episodes (IDs: [40, 41, 42, 43, 44, 45, 46, 47, 48, 49])
  > Total for LaGarNet: 50 episodes

Human:
  - Longsleeve: 10 episodes (IDs: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19])
  - Trousers: 10 episodes (IDs: [30, 31, 32,

In [2]:
# --- Step 2: Formatting Function ---
def fmt(method, step, category):
    """Calculates Mean and Std, formats as 'Mean \pm Std' and formats SR."""
    data = results[method][step][category]
    if data['total'] == 0:
        # Reduced from 5 columns to 4
        return "N/A & N/A & N/A & 0/0"
    
    nc_m, nc_s = np.mean(data['NC']), np.std(data['NC'])
    ni_m, ni_s = np.mean(data['NI']), np.std(data['NI'])
    iou_m, iou_s = np.mean(data['MaxIoU']), np.std(data['MaxIoU'])
    # Replaced sr1 and sr2 with a single sr
    sr = data['success']
    tot = data['total']
    
    # Removed the extra SR output
    return f"{nc_m:.1f} $\pm$ {nc_s:.1f} & {ni_m:.1f} $\pm$ {ni_s:.1f} & {iou_m:.1f} $\pm$ {iou_s:.1f} & {sr}/{tot}"

# --- Step 3: Generate the LaTeX Table ---
latex_str = r"""\begin{table*}[t!]
\caption{Real-world garment flattening performance of different pick-and-place (PnP) controllers... (Insert full caption here)}
\label{tab:real-worldflattening}
\centering
\resizebox{\textwidth}{!}{%
\renewcommand{\arraystretch}{1.2} 
\setlength{\tabcolsep}{2pt}
% Updated column structure: 13 total columns instead of 16
\begin{tabular}{c|cccc|cccc||cccc}
% Updated multicolumn spans from 5 to 4
Method & \multicolumn{4}{c|}{PlaNet-ClothPick \cite{kadi2024planet}}  & \multicolumn{4}{c||}{LaGarNet} & \multicolumn{4}{c}{Human Policy} \\
\cline{1-13} % Updated to reflect 13 columns
% Updated headers to a single SR
10 Steps & NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$     & SR  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR  $\uparrow$ \\
\hline
"""

# Generate 10 Steps Rows
for cat in ['Longsleeve', 'Trousers', 'Dress', 'Skirt']:
    row = f"{cat} & " + " & ".join([fmt(m, 10, cat) for m in ['PlaNet-ClothPick', 'LaGarNet', 'Human']]) + r" \\" + "\n"
    latex_str += row

latex_str += r"\hline" + "\n"
latex_str += f"Real Total & " + " & ".join([fmt(m, 10, 'Real Total') for m in ['PlaNet-ClothPick', 'LaGarNet', 'Human']]) + r" \\" + "\n"

latex_str += r"""\hline
\hline
20 Steps 
% Updated headers to a single SR
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$     & SR  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR  $\uparrow$ \\
\hline
"""

# Generate 20 Steps Rows
for cat in ['Longsleeve', 'Trousers', 'Dress', 'Skirt']:
    row = f"{cat} & " + " & ".join([fmt(m, 20, cat) for m in ['PlaNet-ClothPick', 'LaGarNet', 'Human']]) + r" \\" + "\n"
    latex_str += row

latex_str += r"\hline" + "\n"
latex_str += f"Real Total & " + " & ".join([fmt(m, 20, 'Real Total') for m in ['PlaNet-ClothPick', 'LaGarNet', 'Human']]) + r" \\" + "\n"

latex_str += r"""\bottomrule
\end{tabular}
}
\end{table*}"""

print(latex_str)

\begin{table*}[t!]
\caption{Real-world garment flattening performance of different pick-and-place (PnP) controllers... (Insert full caption here)}
\label{tab:real-worldflattening}
\centering
\resizebox{\textwidth}{!}{%
\renewcommand{\arraystretch}{1.2} 
\setlength{\tabcolsep}{2pt}
% Updated column structure: 13 total columns instead of 16
\begin{tabular}{c|cccc|cccc||cccc}
% Updated multicolumn spans from 5 to 4
Method & \multicolumn{4}{c|}{PlaNet-ClothPick \cite{kadi2024planet}}  & \multicolumn{4}{c||}{LaGarNet} & \multicolumn{4}{c}{Human Policy} \\
\cline{1-13} % Updated to reflect 13 columns
% Updated headers to a single SR
10 Steps & NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$     & SR  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR  $\uparrow$ \\
\hline
Longsleeve & 67.1 $\pm$ 12.7 & 30.4 $\pm$ 20.9 & 63.2 $\pm$ 10.3 & 0/10 & 73.4 $\pm$ 8.2 & 52.0 $\pm$ 16.2 & 66.3 $\pm$ 7

In [ ]:
# ---> NEW: Step 4: Generate Timing LaTeX Table for LaGarNet <---
print("\n--- LAGARNET TIMING TABLE ---")
if len(lagarnet_timings['perception']) > 0:
    mean_perception = np.mean(lagarnet_timings['perception'])
    mean_inference = np.mean(lagarnet_timings['inference'])
    mean_conversion = np.mean(lagarnet_timings['conversion'])
    mean_execution = np.mean(lagarnet_timings['execution'])
    mean_total = mean_perception + mean_inference + mean_conversion + mean_execution

    timing_latex_str = f"""\\begin{{tabular}}{{@{{}}lc@{{}}}}
\\toprule
\\textbf{{System Phase}} & \\textbf{{Time (s)}} \\\\ \\midrule
Perception Time & {mean_perception:.2f} \\\\
Action Inference & {mean_inference:.2f} \\\\
Conversion Time & {mean_conversion:.2f} \\\\
Robot Execution & {mean_execution:.2f} \\\\ \\midrule
\\textbf{{Total Cycle}} & \\textbf{{{mean_total:.2f}}} \\\\ \\bottomrule
\\end{{tabular}}"""
    
    print(timing_latex_str)
else:
    print("No timing data was found for LaGarNet.")


--- LAGARNET TIMING TABLE ---
\begin{tabular}{@{}lc@{}}
\toprule
\textbf{System Phase} & \textbf{Time (s)} \\ \midrule
Perception Time & 8.19 \\
Action Inference & 7.05 \\
Conversion Time & 0.03 \\
Robot Execution & 9.20 \\ \midrule
\textbf{Total Cycle} & \textbf{24.48} \\ \bottomrule
\end{tabular}


In [4]:
# ---> NEW: Step 5: Generate Specific Table for Yellow Dress <---
print("\n--- YELLOW DRESS SPECIFIC TABLE ---")

# Change these variables if you want to target a different method or step
target_method = 'LaGarNet'
target_step = 20
target_category = 'OOD Dress'

data = results[target_method][target_step][target_category]

if data['total'] > 0:
    nc_m, nc_s = np.mean(data['NC']), np.std(data['NC'])
    ni_m, ni_s = np.mean(data['NI']), np.std(data['NI'])
    iou_m, iou_s = np.mean(data['MaxIoU']), np.std(data['MaxIoU'])
    sr = data['success']
    tot = data['total']
    
    yellow_dress_latex = f"""\\begin{{tabular}}{{@{{}}lc@{{}}}}
\\toprule
\\textbf{{Metric}} & \\textbf{{Score}} \\\\ \\midrule
NC & {nc_m:.1f} $\\pm$ {nc_s:.1f} \\\\
NI & {ni_m:.1f} $\\pm$ {ni_s:.1f} \\\\
Max IoU & {iou_m:.1f} $\\pm$ {iou_s:.1f} \\\\
SR & {sr}/{tot} \\\\
\\bottomrule
\\end{{tabular}}%"""
    
    print(yellow_dress_latex)
else:
    print(f"No data found for {target_category} using {target_method} at {target_step} steps.")


--- YELLOW DRESS SPECIFIC TABLE ---
\begin{tabular}{@{}lc@{}}
\toprule
\textbf{Metric} & \textbf{Score} \\ \midrule
NC & 77.2 $\pm$ 6.5 \\
NI & 42.1 $\pm$ 22.6 \\
Max IoU & 66.4 $\pm$ 3.2 \\
SR & 0/10 \\
\bottomrule
\end{tabular}%
